In [ ]:
#FOR SAVING PLOTS
import os

# -------------------------------------------------
# OUTPUT FOLDER
# -------------------------------------------------
OUTPUT_DIR = r"..\..\data\exports"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# -------------------------------------------------
# FIGURE SAVING FUNCTION
# -------------------------------------------------
def save_fig(filename, folder=OUTPUT_DIR, dpi=300):
    """
    Saves the current matplotlib figure safely.
    MUST be called BEFORE plt.show().
    """
    path = os.path.join(folder, filename)

    fig = plt.gcf()  # grab current figure explicitly
    fig.savefig(path, dpi=dpi, bbox_inches="tight")

    print(f"Saved figure → {path}")


# -------------------------------------------------
# DYNAMIC FILENAME BUILDER
# -------------------------------------------------
def make_filename(site_name, years, months, mode):
    site_name = site_name.replace(" ", "_")

    year_str = "-".join(years) if isinstance(years, list) else str(years)
    month_str = "-".join(months) if isinstance(months, list) else str(months)

    return f"{site_name}_wind_{year_str}_{month_str}_{mode}.png"

In [ ]:
#BEFORE EVERY PLOT
import sys
import os
import glob
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

# allow config import from project root
PROJECT_ROOT = os.path.abspath(r"..\..")
sys.path.append(PROJECT_ROOT)

from config import SITES, VARIABLES, TIME_DIM

In [ ]:


# =====================================================
# CONFIG (YOU CONTROL EVERYTHING HERE)
# =====================================================

site = SITES["hanle"]     # change site here

selected_years = ["2025"]     # or ["all"]
selected_months = ["all"] # or ["all"]

plot_mode = "combined"  
# options: "combined", "separate"

# =====================================================
# LOAD DATA
# =====================================================
files = sorted(glob.glob(site["path"]))

print("Files found:", len(files))

if len(files) == 0:
    raise ValueError("No files found. Check config path.")

ds = xr.open_mfdataset(files, combine="by_coords")
ds = ds.sortby(TIME_DIM)

time = ds[TIME_DIM].values

# =====================================================
# FILTER TIME (YEAR + MONTH)
# =====================================================
mask = []

for t in time:
    year = str(t)[:4]
    month = str(t)[5:7]

    ok_year = ("all" in selected_years) or (year in selected_years)
    ok_month = ("all" in selected_months) or (month in selected_months)

    mask.append(ok_year and ok_month)

mask = np.array(mask)

ds_f = ds.isel({TIME_DIM: mask})
time_f = ds_f[TIME_DIM]

# =====================================================
# WIND EXTRACTION (NEAREST GRID POINT)
# =====================================================
u = ds_f["u10"].sel(
    latitude=site["lat"],
    longitude=site["lon"],
    method="nearest"
)

v = ds_f["v10"].sel(
    latitude=site["lat"],
    longitude=site["lon"],
    method="nearest"
)

# =====================================================
# WIND SPEED + DIRECTION
# =====================================================
speed = np.sqrt(u.values**2 + v.values**2)
direction = (270 - np.degrees(np.arctan2(v, u))) % 360
theta = np.radians(direction)

# =====================================================
# PLOTTING
# =====================================================

# -----------------------------
# MODE 1: COMBINED PLOT
# -----------------------------
if plot_mode == "combined":

    plt.figure(figsize=(6,6))
    ax = plt.subplot(111, polar=True)

    ax.scatter(
        theta,
        speed,
        c=speed,
        cmap="viridis",
        s=8,
        alpha=0.7
    )

    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)

    ax.set_title(
        f"{site['name']} | Years: {selected_years} | Months: {selected_months}"
    )

    plt.colorbar(ax.collections[0], ax=ax, pad=0.1, label="Wind Speed (m/s)")
  
    # filename = make_filename(
    #     site["name"],
    #     selected_years,
    #     selected_months,
    #     plot_mode
    # )

    # save_fig(filename)
    plt.show()
    


# -----------------------------
# MODE 2: SEPARATE PLOTS BY MONTH
# -----------------------------
elif plot_mode == "separate":

    for m in selected_months:

        idx = np.array([str(t)[5:7] == m for t in time_f.values])

        if np.sum(idx) == 0:
            continue

        plt.figure(figsize=(6,6))
        ax = plt.subplot(111, polar=True)

        ax.scatter(
            theta[idx],
            speed[idx],
            c=speed[idx],
            cmap="viridis",
            s=8,
            alpha=0.7
        )

        ax.set_theta_zero_location("N")
        ax.set_theta_direction(-1)

        ax.set_title(
            f"{site['name']} | {selected_years} - {m}"
        )

        plt.colorbar(ax.collections[0], ax=ax, pad=0.1, label="Wind Speed (m/s)")
        # filename = make_filename(
    #     site["name"],
    #     selected_years,
    #     selected_months,
    #     plot_mode
    # )

    # save_fig(filename)
        plt.show()


else:
    raise ValueError("plot_mode must be 'combined' or 'separate'")